In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from typing import List, Dict
import os
import sys
from urllib.parse import urljoin
from pprint import pprint
import re

BASE_PATH = "../../data/RAG"

In [2]:
def get_maple_guide(url: str) -> str:
    """메이플 가이드라인 문서 크롤링

    Args:
        page (int): 페이지 번호

    Returns:
        str: 웹페이지의 HTML 내용
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()  # 오류가 있으면 예외를 발생시킴
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"페이지 요청 중 에러 발생: {e}")
        return ""

In [3]:
def get_guide_list(board_id: int):
    url = "https://maplestory.nexon.com/guide/n23gameinformation/articles"

    params = {
        "boardId": board_id
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "Chrome/120.0.0.0"
        )
    }

    response = requests.post(
        url,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    return response.json()

In [4]:
data = get_guide_list(429467339)

pprint(data["pageInfo"])

{'numberOfElements': 13, 'totalElements': 13, 'totalPages': 1}


In [5]:
for item in data["list"]:
    print(
        item["articleId"],
        item["title"]
    )

410 [장비] 기본 가이드
411 [장비] 추가 옵션
374 [장비] 주문서 강화
412 [장비] 스타포스 강화
413 [장비] 잠재능력/에디셔널 잠재능력
414 [장비] 장비 전승
415 [장비] 세트 아이템
416 [장비] 소울 웨폰
418 소비 아이템
419 기타 아이템
420 설치 아이템
421 캐시 아이템
422 치장 아이템


In [ ]:
def parse_api_guide(item: dict) -> dict:

    article_id = item["articleId"]

    return {
        "article_id": article_id,
        "title": item["title"],
        "content": item["textContent"],
        "url": (
            "https://maplestory.nexon.com/"
            f"Guide/N23GameInformation/Articles/{article_id}"
        )
    }

In [17]:
board_ids = {
    "기초 가이드": 429467337,
    "성장": 429467338,
    "아이템": 429467339,
    "사냥/보스 컨텐츠": 429467340,
    "스페셜 컨텐츠": 429467341,
    "커뮤니티": 429467342,
    "거래": 429467343,
    "캐시 & 코디": 429467344,
    "기타/TIP": 429467345,
}

all_guides = []

for category, board_id in board_ids.items():
    data = get_guide_list(board_id)
    for item in data['list']:

        guide = parse_api_guide(item)
        guide['category'] = category
        guide['board_id'] = board_id

        all_guides.append(guide)

print("수집 문서 개수:", len(all_guides))


수집 문서 개수: 106


In [22]:
def save_guides_to_json(guides, BASE_PATH, file_name="maple_guides.json"):

    file_path = os.path.join(BASE_PATH, file_name)

    with open(file_path, "w", encoding='utf-8') as f:
        json.dump(guides, f, ensure_ascii=False, indent=2)

    print("JSON 저장 완료")
    return file_path

save_guides_to_json(all_guides, BASE_PATH)

JSON 저장 완료


'../../data/RAG\\maple_guides.json'